# Experiment 2.10: Anti-Sharpness — Muon Finds Flatter Minima via Hessian Trace Analysis

## Scientific Motivation

A central puzzle in optimization theory is the **sharpness-generalization connection**: models that converge to *flatter* minima (lower curvature of the loss surface) tend to generalize better. The Hessian matrix $H$ of the loss function encodes local curvature, and its trace $\text{tr}(H) = \sum_i \lambda_i$ measures **total curvature** across all parameter directions.

**Prior result (Experiment 1.3b-ii-A)** showed that Muon's Newton-Schulz orthogonalized gradients project **11x more** onto gauge-Hessian directions — the directions associated with the continuous symmetry group of deep linear networks. This is seemingly paradoxical: if Muon aggressively moves along *sharp* directions, shouldn't it end up in *sharper* minima?

## The Anti-Sharpness Hypothesis

We hypothesize the opposite: **aggressive movement *through* sharp directions helps the optimizer *escape* them**, rather than settling into them. The mechanism is analogous to how a ball rolling fast through a narrow valley will overshoot and find a wider basin, while a slow ball gets trapped.

$$\boxed{\text{Hypothesis: } \text{tr}(H_{\text{Muon}}) < 0.5 \cdot \text{tr}(H_{\text{SGD}}) \text{ at convergence}}$$

## Experimental Design

| Component | Choice | Rationale |
|-----------|--------|-----------|
| **Architecture** | 2-layer and 3-layer deep linear networks (4x4) | Small enough for *exact* full Hessian computation; deep linear nets have known gauge symmetries $W_2 W_1 = (W_2 A^{-1})(A W_1)$ |
| **Parameter count** | 32 (2-layer) and 48 (3-layer) | Full Hessian is $32 \times 32$ or $48 \times 48$ — computationally tractable |
| **Hessian method** | Central finite differences on the gradient | Gives symmetric Hessian to machine precision; avoids autograd dependency |
| **Repetitions** | 10 seeds per architecture | Statistical robustness for mean and std estimates |
| **Convergence criterion** | Loss < 0.001 × initial loss | Both optimizers reach equivalent quality solutions before Hessian comparison |

## Key Measurements

- **$\text{tr}(H)$**: Total curvature (sum of all eigenvalues) — primary hypothesis metric
- **$\lambda_{\max}$**: Sharpness — largest eigenvalue, dominant curvature direction
- **Effective Hessian rank**: Number of eigenvalues $> 0.01 \cdot \lambda_{\max}$ — how many directions have significant curvature
- **Gauge-flat count**: Number of eigenvalues $< 0.001 \cdot \lambda_{\max}$ — directions with near-zero curvature (gauge symmetries)
- **Condition number $\kappa(H)$**: $\lambda_{\max} / \lambda_{\min,+}$ — ratio of strongest to weakest positive curvature

## Connection to RG Gauge-Fixing Theory

In the renormalization group (RG) interpretation of Muon, the Newton-Schulz orthogonalization acts as a **gauge-fixing** operation that removes redundant degrees of freedom from the gradient. If this interpretation is correct, Muon should systematically eliminate curvature along gauge-flat directions, yielding a Hessian spectrum that is both *flatter* (lower trace) and *lower-rank* (more near-zero eigenvalues) than SGD's.

In [ ]:
import numpy as np
import os
import sys

print("Environment:")
print(f"  NumPy version: {np.__version__}")
print(f"  Python version: {sys.version.split()[0]}")
print(f"  Working directory: {os.getcwd()}")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

## Hyperparameter Configuration

The learning rates are chosen so that both optimizers reach convergence reliably within 3000 steps. Muon uses a slightly higher learning rate (`LR_MUON = 0.02` vs `LR_SGD = 0.01`) because the Newton-Schulz orthogonalization normalizes gradient magnitudes, making each step a fixed-norm rotation rather than a magnitude-dependent translation. The convergence factor of 0.001 ensures both optimizers reach solutions of comparable quality before we probe the Hessian — we are comparing the *geometry* of the minima they find, not their training speed.

The Hessian finite-difference step `eps = 1e-5` balances truncation error (favors small eps) against floating-point cancellation error (favors large eps). For double-precision arithmetic, `1e-5` is near-optimal for central differences.

In [ ]:
DIM = 4                 # 4x4 matrices => 16 params per layer
NUM_STEPS = 3000        # Max training steps
LR_SGD = 0.01
LR_MUON = 0.02
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 10
CONVERGENCE_FACTOR = 0.001  # Stop when loss < this fraction of initial loss
HESSIAN_EPS = 1e-5          # Finite difference step for Hessian

print("Experiment Configuration:")
print(f"  Matrix dimension:        {DIM}x{DIM} => {DIM*DIM} params per layer")
print(f"  Max training steps:      {NUM_STEPS}")
print(f"  SGD learning rate:       {LR_SGD}")
print(f"  Muon learning rate:      {LR_MUON}")
print(f"  Momentum (both):         {MOMENTUM}")
print(f"  Newton-Schulz iters:     {NS_ITERS}")
print(f"  Number of seeds:         {NUM_SEEDS}")
print(f"  Convergence factor:      {CONVERGENCE_FACTOR} (stop at loss < {CONVERGENCE_FACTOR} * initial_loss)")
print(f"  Hessian FD epsilon:      {HESSIAN_EPS}")
print()
print(f"  2-layer net: {2 * DIM * DIM} total params => {2 * DIM * DIM}x{2 * DIM * DIM} Hessian")
print(f"  3-layer net: {3 * DIM * DIM} total params => {3 * DIM * DIM}x{3 * DIM * DIM} Hessian")

## Network Utilities — Deep Linear Networks

A deep linear network computes $Y = W_L \cdots W_2 W_1 X$. Despite being mathematically equivalent to a single linear map $W_{\text{eff}} = \prod_i W_i$, the factored parameterization introduces a **rich loss landscape** with:

- **Gauge symmetries**: For any invertible $A$, the transformation $W_{i+1} \to W_{i+1} A^{-1}, W_i \to A W_i$ preserves the end-to-end map. This creates continuous families of equivalent solutions — flat valleys in the loss landscape.
- **Saddle points**: The loss surface has saddle points (not local minima) that can trap slow optimizers.
- **Non-convexity**: Even though the mapping is linear, the loss as a function of the *factored* weights is non-convex.

Weights are initialized near identity ($I + 0.1 \cdot \mathcal{N}(0,1)$) to start in a regime where the network is close to computing the identity transform, ensuring stable initial gradients.

In [ ]:
def init_weights(num_layers, dim, seed):
    """Initialize layers near identity."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(dim) + rng.randn(dim, dim) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
def forward_linear(weights, X):
    """Forward pass through deep linear net."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y_target):
    """MSE loss."""
    Y_pred = forward_linear(weights, X)
    diff = Y_pred - Y_target
    return 0.5 * np.mean(diff ** 2)

In [ ]:
def compute_gradients(weights, X, Y_target):
    """Backprop through deep linear net."""
    num_layers = len(weights)
    batch_size = X.shape[1]

    # Forward pass — store activations
    activations = [X.copy()]
    for W in weights:
        activations.append(W @ activations[-1])

    # Output error
    Y_pred = activations[-1]
    delta = (Y_pred - Y_target) / batch_size  # shape (dim, batch)

    # Backward pass
    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        grads[l] = delta @ activations[l].T  # (dim, dim)
        if l > 0:
            delta = weights[l].T @ delta

    return grads

## Newton-Schulz Iteration — Muon's Core Mechanism

The Newton-Schulz (NS) iteration computes the **orthogonal polar factor** of a matrix $G = U \Sigma V^T$ by iteratively converging to $U V^T$. Starting from $X_0 = G / \|G\|_F$ and iterating:

$$X_{k+1} = \frac{3}{2} X_k - \frac{1}{2} X_k X_k^T X_k$$

this converges cubically to the nearest orthogonal matrix (in Frobenius norm). After 5 iterations, the result is an orthogonal matrix that preserves the *direction* information of $G$ while discarding all *magnitude* (singular value) information.

**Why this matters for sharpness**: By replacing the gradient $G$ with its orthogonal polar factor $UV^T$, Muon takes steps of **equal magnitude in every singular direction**. This means sharp directions (large Hessian eigenvalues, which produce large gradient components) receive *no more* step magnitude than flat directions. The optimizer is effectively "blind" to curvature, which prevents it from being attracted to sharp valleys where the gradient is large.

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=NS_ITERS):
    """Apply Newton-Schulz iteration to approximate orthogonal polar factor."""
    # Normalize
    norm = np.linalg.norm(G, ord='fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        A = X @ X.T
        # X <- 1.5*X - 0.5*A@X  (Newton-Schulz iteration for polar decomposition)
        X = 1.5 * X - 0.5 * A @ X

    return X

## Optimizers — SGD with Momentum vs. Muon

Both optimizers use the same momentum coefficient ($\beta = 0.9$) to ensure any differences in the converged Hessian spectrum are attributable to the **gradient preprocessing** (identity for SGD, Newton-Schulz orthogonalization for Muon), not to differences in momentum dynamics.

**SGD with momentum** follows the standard update:
$$v_{t+1} = \beta v_t + \nabla L(W_t), \quad W_{t+1} = W_t - \eta \cdot v_t$$

**Muon** replaces $\nabla L$ with its orthogonal polar factor:
$$v_{t+1} = \beta v_t + \text{NS}(\nabla L(W_t)), \quad W_{t+1} = W_t - \eta \cdot v_t$$

Both optimizers start from **identical initializations** (same seed + 1000 offset) to ensure a fair comparison. The convergence criterion stops training when the loss drops below $0.001 \times$ the initial loss, ensuring both optimizers have reached solutions of comparable quality before we examine the local geometry.

In [ ]:
def train_sgd(weights, X, Y_target, lr, num_steps, convergence_threshold):
    """Train with SGD + momentum."""
    velocities = [np.zeros_like(W) for W in weights]
    initial_loss = compute_loss(weights, X, Y_target)
    target_loss = initial_loss * convergence_threshold

    for step in range(num_steps):
        loss = compute_loss(weights, X, Y_target)
        if loss < target_loss:
            return weights, loss, step, True

        grads = compute_gradients(weights, X, Y_target)
        for i in range(len(weights)):
            velocities[i] = MOMENTUM * velocities[i] + grads[i]
            weights[i] = weights[i] - lr * velocities[i]

    final_loss = compute_loss(weights, X, Y_target)
    return weights, final_loss, num_steps, (final_loss < target_loss)

In [ ]:
def train_muon(weights, X, Y_target, lr, num_steps, convergence_threshold):
    """Train with Muon (NS-orthogonalized gradients + momentum)."""
    velocities = [np.zeros_like(W) for W in weights]
    initial_loss = compute_loss(weights, X, Y_target)
    target_loss = initial_loss * convergence_threshold

    for step in range(num_steps):
        loss = compute_loss(weights, X, Y_target)
        if loss < target_loss:
            return weights, loss, step, True

        grads = compute_gradients(weights, X, Y_target)
        for i in range(len(weights)):
            # Newton-Schulz orthogonalization of gradient
            G_orth = newton_schulz_orthogonalize(grads[i])
            velocities[i] = MOMENTUM * velocities[i] + G_orth
            weights[i] = weights[i] - lr * velocities[i]

    final_loss = compute_loss(weights, X, Y_target)
    return weights, final_loss, num_steps, (final_loss < target_loss)

## Full Hessian Computation via Central Finite Differences

Computing the full Hessian matrix $H_{ij} = \frac{\partial^2 L}{\partial \theta_i \partial \theta_j}$ is the most computationally expensive part of this experiment. For $n$ parameters, we need $2n$ gradient evaluations (one forward and one backward perturbation per parameter dimension).

**Method**: Central finite differences on the gradient vector:
$$H_{:,i} = \frac{\nabla L(\theta + \epsilon e_i) - \nabla L(\theta - \epsilon e_i)}{2\epsilon}$$

This gives $O(\epsilon^2)$ accuracy (vs. $O(\epsilon)$ for forward differences) and produces a naturally symmetric matrix after averaging $H \leftarrow \frac{1}{2}(H + H^T)$.

**Cost**: For 32 parameters (2-layer), we compute $2 \times 32 = 64$ gradient evaluations, each involving a forward and backward pass. For 48 parameters (3-layer), it is $2 \times 48 = 96$ gradient evaluations. This is tractable only because of the small network size.

In [ ]:
def weights_to_vector(weights):
    """Flatten all weight matrices into a single vector."""
    return np.concatenate([W.flatten() for W in weights])

In [ ]:
def vector_to_weights(vec, shapes):
    """Unflatten a vector back into list of weight matrices."""
    weights = []
    idx = 0
    for shape in shapes:
        size = shape[0] * shape[1]
        W = vec[idx:idx + size].reshape(shape)
        weights.append(W)
        idx += size
    return weights

In [ ]:
def compute_gradient_vector(weights, X, Y_target):
    """Return gradient as a flat vector."""
    grads = compute_gradients(weights, X, Y_target)
    return np.concatenate([g.flatten() for g in grads])

In [ ]:
def compute_full_hessian(weights, X, Y_target, eps=HESSIAN_EPS):
    """Compute full Hessian via central finite differences on the gradient."""
    shapes = [W.shape for W in weights]
    theta = weights_to_vector(weights)
    n_params = len(theta)

    H = np.zeros((n_params, n_params))

    for i in range(n_params):
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[i] += eps
        theta_minus[i] -= eps

        w_plus = vector_to_weights(theta_plus, shapes)
        w_minus = vector_to_weights(theta_minus, shapes)

        grad_plus = compute_gradient_vector(w_plus, X, Y_target)
        grad_minus = compute_gradient_vector(w_minus, X, Y_target)

        H[:, i] = (grad_plus - grad_minus) / (2 * eps)

    # Symmetrize
    H = 0.5 * (H + H.T)
    return H

## Hessian Spectral Analysis

The eigenvalue spectrum of the Hessian at a converged solution reveals the **geometric character** of the minimum:

- **Trace** $\text{tr}(H) = \sum \lambda_i$: Total curvature. Lower trace = flatter minimum overall. This is our primary hypothesis metric.
- **$\lambda_{\max}$**: The sharpest direction. High $\lambda_{\max}$ means there exists at least one direction of extreme curvature, which can destabilize training and hurt generalization.
- **Effective rank**: How many directions carry significant curvature. Lower effective rank = the loss landscape is more "degenerate" (many flat directions), which is characteristic of overparameterized models finding gauge-symmetric solutions.
- **Gauge-flat count**: Directions with near-zero curvature ($< 0.001 \lambda_{\max}$). In deep linear networks, these correspond to the gauge symmetry orbits. More gauge-flat directions at convergence suggests the optimizer has found a solution deeper in the gauge manifold.
- **Condition number** $\kappa = \lambda_{\max} / \lambda_{\min,+}$: Ratio of strongest to weakest positive curvature. High $\kappa$ means the loss landscape is highly anisotropic — some directions are much sharper than others.

We also track **negative eigenvalues**, which would indicate the optimizer has converged to a saddle point rather than a true minimum. A small number of mildly negative eigenvalues can arise from finite-difference numerical error.

In [ ]:
def analyze_hessian(H):
    """Compute all Hessian statistics."""
    eigenvalues = np.linalg.eigvalsh(H)
    eigenvalues_sorted = np.sort(eigenvalues)[::-1]  # descending

    lambda_max = eigenvalues_sorted[0]
    trace_H = np.sum(eigenvalues)

    # Effective rank: eigenvalues > 0.01 * lambda_max
    if lambda_max > 1e-15:
        eff_rank = np.sum(np.abs(eigenvalues) > 0.01 * lambda_max)
        gauge_flat = np.sum(np.abs(eigenvalues) < 0.001 * lambda_max)
    else:
        eff_rank = 0
        gauge_flat = len(eigenvalues)

    # Condition number: lambda_max / lambda_min_positive
    positive_eigs = eigenvalues[eigenvalues > 1e-12]
    if len(positive_eigs) > 0:
        kappa = lambda_max / np.min(positive_eigs)
    else:
        kappa = np.inf

    # Also track negative eigenvalue count and magnitude
    neg_eigs = eigenvalues[eigenvalues < -1e-12]
    n_negative = len(neg_eigs)
    sum_negative = np.sum(neg_eigs) if n_negative > 0 else 0.0

    return {
        'trace': trace_H,
        'lambda_max': lambda_max,
        'eff_rank': eff_rank,
        'gauge_flat': gauge_flat,
        'kappa': kappa,
        'n_negative': n_negative,
        'sum_negative': sum_negative,
        'eigenvalues': eigenvalues_sorted,
    }

## Main Experiment — Running the Full Comparison Suite

The experiment runs in two phases:

**Part A: 2-Layer Net (32 parameters)**
- Gauge group: $GL(4)$ has 16 dimensions, so we expect ~16 gauge-flat directions in the Hessian
- The 32x32 Hessian is small enough that eigendecomposition is essentially free

**Part B: 3-Layer Net (48 parameters)**
- Gauge group: $GL(4) \times GL(4)$ has 32 dimensions, so we expect ~32 gauge-flat directions
- More gauge symmetry means more opportunity for Muon's orthogonalization to differentiate itself from SGD
- **Prediction**: The anti-sharpness effect should be *stronger* in the 3-layer case because there are more gauge directions for Muon to navigate through

Each phase runs 10 independent seeds with different random targets and data, starting both optimizers from the same initialization per seed. After convergence, the full Hessian is computed and analyzed.

In [ ]:
def run_single_experiment(num_layers, dim, seed):
    """Run one comparison between SGD and Muon for a given architecture and seed."""
    # Generate random target and data
    rng = np.random.RandomState(seed)
    W_target_list = [rng.randn(dim, dim) * 0.3 for _ in range(num_layers)]
    X = rng.randn(dim, 32) * 0.5  # 32 data points

    # Compute target output
    Y_target = X.copy()
    for W in W_target_list:
        Y_target = W @ Y_target

    # Print initial diagnostic for this seed
    initial_loss_check = compute_loss(init_weights(num_layers, dim, seed + 1000), X, Y_target)
    print(f"    [Seed {seed}] Initial loss: {initial_loss_check:.4e}, target convergence loss: {initial_loss_check * CONVERGENCE_FACTOR:.4e}")

    # Train SGD
    w_sgd = init_weights(num_layers, dim, seed + 1000)
    w_sgd, loss_sgd, steps_sgd, conv_sgd = train_sgd(
        w_sgd, X, Y_target, LR_SGD, NUM_STEPS, CONVERGENCE_FACTOR)

    # Train Muon (start from SAME initialization)
    w_muon = init_weights(num_layers, dim, seed + 1000)
    w_muon, loss_muon, steps_muon, conv_muon = train_muon(
        w_muon, X, Y_target, LR_MUON, NUM_STEPS, CONVERGENCE_FACTOR)

    # Print convergence comparison for this seed
    print(f"    [Seed {seed}] SGD:  loss={loss_sgd:.4e}, steps={steps_sgd:4d}, converged={'YES' if conv_sgd else 'NO '}")
    print(f"    [Seed {seed}] Muon: loss={loss_muon:.4e}, steps={steps_muon:4d}, converged={'YES' if conv_muon else 'NO '}")

    # Compute full Hessian at convergence
    H_sgd = compute_full_hessian(w_sgd, X, Y_target)
    H_muon = compute_full_hessian(w_muon, X, Y_target)

    # Print Hessian basic diagnostics
    n_params = num_layers * dim * dim
    print(f"    [Seed {seed}] Hessian shape: {H_sgd.shape}, symmetry check (SGD): max|H-H^T| = {np.max(np.abs(H_sgd - H_sgd.T)):.2e}")
    print(f"    [Seed {seed}] Hessian shape: {H_muon.shape}, symmetry check (Muon): max|H-H^T| = {np.max(np.abs(H_muon - H_muon.T)):.2e}")

    stats_sgd = analyze_hessian(H_sgd)
    stats_muon = analyze_hessian(H_muon)

    # Print per-seed Hessian trace comparison
    ratio = stats_muon['trace'] / stats_sgd['trace'] if abs(stats_sgd['trace']) > 1e-15 else float('inf')
    print(f"    [Seed {seed}] tr(H): SGD={stats_sgd['trace']:.4e}, Muon={stats_muon['trace']:.4e}, ratio={ratio:.4f}")
    print(f"    [Seed {seed}] lambda_max: SGD={stats_sgd['lambda_max']:.4e}, Muon={stats_muon['lambda_max']:.4e}")
    print(f"    [Seed {seed}] Negative eigenvalues: SGD={stats_sgd['n_negative']}, Muon={stats_muon['n_negative']}")
    print()

    return {
        'sgd': {**stats_sgd, 'loss': loss_sgd, 'steps': steps_sgd, 'converged': conv_sgd},
        'muon': {**stats_muon, 'loss': loss_muon, 'steps': steps_muon, 'converged': conv_muon},
    }

In [ ]:
def print_separator(char='=', width=100):
    print(char * width)

In [ ]:
def run_experiment_suite(num_layers, dim, label):
    """Run full experiment suite for a given architecture."""
    n_params = num_layers * dim * dim

    print_separator()
    print(f"  {label}: {num_layers}-LAYER DEEP LINEAR NET ({dim}x{dim}, {n_params} params)")
    print_separator()
    print()

    # Print theoretical gauge dimension
    n_gauge = (num_layers - 1) * dim * dim
    print(f"  Theoretical gauge group dimension: {n_gauge} (out of {n_params} total params)")
    print(f"  Expected gauge-flat fraction: {n_gauge}/{n_params} = {n_gauge/n_params:.1%}")
    print(f"  Expected non-gauge (physically meaningful) directions: {n_params - n_gauge}")
    print()

    all_results = []
    for i in range(NUM_SEEDS):
        seed = 42 + i * 137
        result = run_single_experiment(num_layers, dim, seed)
        all_results.append(result)
        sgd_c = "YES" if result['sgd']['converged'] else "NO"
        muon_c = "YES" if result['muon']['converged'] else "NO"
        print(f"  Seed {i+1:2d}/{NUM_SEEDS}: SGD conv={sgd_c} (loss={result['sgd']['loss']:.2e}, steps={result['sgd']['steps']})"
              f"  |  Muon conv={muon_c} (loss={result['muon']['loss']:.2e}, steps={result['muon']['steps']})")

    # Aggregate statistics
    metrics = ['trace', 'lambda_max', 'eff_rank', 'gauge_flat', 'kappa']
    metric_labels = {
        'trace': 'tr(H)',
        'lambda_max': 'lambda_max',
        'eff_rank': 'Eff. Hessian rank',
        'gauge_flat': 'Gauge-flat count',
        'kappa': 'Condition number kappa',
    }

    sgd_stats = {m: [] for m in metrics}
    muon_stats = {m: [] for m in metrics}

    for r in all_results:
        for m in metrics:
            sgd_stats[m].append(r['sgd'][m])
            muon_stats[m].append(r['muon'][m])

    # Also gather convergence info
    sgd_losses = [r['sgd']['loss'] for r in all_results]
    muon_losses = [r['muon']['loss'] for r in all_results]
    sgd_conv_count = sum(1 for r in all_results if r['sgd']['converged'])
    muon_conv_count = sum(1 for r in all_results if r['muon']['converged'])
    sgd_steps = [r['sgd']['steps'] for r in all_results]
    muon_steps = [r['muon']['steps'] for r in all_results]

    print()
    print_separator('-')
    print(f"  CONVERGENCE SUMMARY:")
    print(f"    SGD converged:  {sgd_conv_count}/{NUM_SEEDS}  (mean loss = {np.mean(sgd_losses):.2e} +/- {np.std(sgd_losses):.2e})")
    print(f"    Muon converged: {muon_conv_count}/{NUM_SEEDS}  (mean loss = {np.mean(muon_losses):.2e} +/- {np.std(muon_losses):.2e})")
    print(f"    SGD mean steps to converge:  {np.mean(sgd_steps):.0f} +/- {np.std(sgd_steps):.0f}")
    print(f"    Muon mean steps to converge: {np.mean(muon_steps):.0f} +/- {np.std(muon_steps):.0f}")
    if np.mean(muon_steps) > 0:
        print(f"    Speed ratio (SGD steps / Muon steps): {np.mean(sgd_steps)/np.mean(muon_steps):.2f}x")
    print_separator('-')
    print()

    # Print loss quality check
    print(f"  LOSS QUALITY CHECK (are both optimizers at comparable solutions?):")
    print(f"    Max SGD loss:  {np.max(sgd_losses):.2e}")
    print(f"    Max Muon loss: {np.max(muon_losses):.2e}")
    loss_ratio = np.mean(muon_losses) / np.mean(sgd_losses) if np.mean(sgd_losses) > 0 else float('inf')
    print(f"    Mean loss ratio (Muon/SGD): {loss_ratio:.2f}")
    if loss_ratio > 10 or loss_ratio < 0.1:
        print(f"    WARNING: Large loss ratio suggests optimizers converged to different quality solutions!")
        print(f"    Hessian comparison may not be apples-to-apples.")
    else:
        print(f"    OK: Both optimizers at comparable solution quality.")
    print()

    # Print comparison table
    print(f"  {'Metric':<28s} | {'SGD (mean +/- std)':<28s} | {'Muon (mean +/- std)':<28s} | {'Ratio Muon/SGD':<16s}")
    print(f"  {'-'*28}-+-{'-'*28}-+-{'-'*28}-+-{'-'*16}")

    ratios = {}
    for m in metrics:
        sgd_mean = np.mean(sgd_stats[m])
        sgd_std = np.std(sgd_stats[m])
        muon_mean = np.mean(muon_stats[m])
        muon_std = np.std(muon_stats[m])

        if abs(sgd_mean) > 1e-15:
            ratio = muon_mean / sgd_mean
            ratio_str = f"{ratio:.4f}"
        else:
            ratio = np.inf
            ratio_str = "N/A"

        ratios[m] = ratio
        label_str = metric_labels[m]

        # Format numbers appropriately
        if m in ['eff_rank', 'gauge_flat']:
            sgd_str = f"{sgd_mean:8.1f} +/- {sgd_std:6.1f}"
            muon_str = f"{muon_mean:8.1f} +/- {muon_std:6.1f}"
        elif m == 'kappa':
            sgd_str = f"{sgd_mean:10.1f} +/- {sgd_std:8.1f}"
            muon_str = f"{muon_mean:10.1f} +/- {muon_std:8.1f}"
        else:
            sgd_str = f"{sgd_mean:12.4e} +/- {sgd_std:.2e}"
            muon_str = f"{muon_mean:12.4e} +/- {muon_std:.2e}"

        print(f"  {label_str:<28s} | {sgd_str:<28s} | {muon_str:<28s} | {ratio_str:<16s}")

    print()

    # Print interpretation of each metric
    print(f"  METRIC-BY-METRIC INTERPRETATION:")
    print(f"    tr(H) ratio = {ratios['trace']:.4f}: Muon's total curvature is {'LOWER' if ratios['trace'] < 1 else 'HIGHER'} than SGD's by {abs(1-ratios['trace'])*100:.1f}%")
    print(f"    lambda_max ratio = {ratios['lambda_max']:.4f}: Muon's sharpest direction is {'LESS' if ratios['lambda_max'] < 1 else 'MORE'} sharp by {abs(1-ratios['lambda_max'])*100:.1f}%")
    print(f"    Eff. rank ratio = {ratios['eff_rank']:.4f}: Muon has {'FEWER' if ratios['eff_rank'] < 1 else 'MORE'} significantly curved directions")
    print(f"    Gauge-flat ratio = {ratios['gauge_flat']:.4f}: Muon has {'MORE' if ratios['gauge_flat'] > 1 else 'FEWER'} near-zero curvature directions")
    if ratios['kappa'] != np.inf:
        print(f"    Kappa ratio = {ratios['kappa']:.4f}: Muon's landscape is {'LESS' if ratios['kappa'] < 1 else 'MORE'} anisotropic")
    print()

    # KEY HYPOTHESIS TEST
    print_separator('*')
    trace_ratio = ratios['trace']
    print(f"  KEY HYPOTHESIS TEST: tr(H_Muon) < 0.5 * tr(H_SGD)")
    print(f"  Ratio tr(H_Muon)/tr(H_SGD) = {trace_ratio:.4f}")
    if trace_ratio < 0.5:
        print(f"  >>> HYPOTHESIS CONFIRMED: Muon finds significantly flatter minima (ratio = {trace_ratio:.4f} < 0.5)")
    elif trace_ratio < 1.0:
        print(f"  >>> PARTIAL SUPPORT: Muon finds flatter minima (ratio = {trace_ratio:.4f} < 1.0) but not 2x flatter")
    else:
        print(f"  >>> HYPOTHESIS REJECTED: Muon does NOT find flatter minima (ratio = {trace_ratio:.4f} >= 1.0)")
    print_separator('*')
    print()

    # Additional analysis: eigenvalue spectrum comparison
    print(f"  EIGENVALUE SPECTRUM COMPARISON (averaged over {NUM_SEEDS} seeds):")
    print(f"  {'Bin':<30s} | {'SGD count':<14s} | {'Muon count':<14s}")
    print(f"  {'-'*30}-+-{'-'*14}-+-{'-'*14}")

    # Compute averaged eigenvalue distributions
    sgd_all_eigs = np.array([r['sgd']['eigenvalues'] for r in all_results])
    muon_all_eigs = np.array([r['muon']['eigenvalues'] for r in all_results])

    # Bins for eigenvalue magnitudes
    bins = [
        ('|eig| > 1.0', lambda e: np.sum(np.abs(e) > 1.0)),
        ('0.1 < |eig| <= 1.0', lambda e: np.sum((np.abs(e) > 0.1) & (np.abs(e) <= 1.0))),
        ('0.01 < |eig| <= 0.1', lambda e: np.sum((np.abs(e) > 0.01) & (np.abs(e) <= 0.1))),
        ('0.001 < |eig| <= 0.01', lambda e: np.sum((np.abs(e) > 0.001) & (np.abs(e) <= 0.01))),
        ('|eig| <= 0.001', lambda e: np.sum(np.abs(e) <= 0.001)),
        ('eig < 0 (negative)', lambda e: np.sum(e < -1e-12)),
    ]

    for bin_label, bin_fn in bins:
        sgd_counts = [bin_fn(sgd_all_eigs[i]) for i in range(NUM_SEEDS)]
        muon_counts = [bin_fn(muon_all_eigs[i]) for i in range(NUM_SEEDS)]
        print(f"  {bin_label:<30s} | {np.mean(sgd_counts):6.1f} +/- {np.std(sgd_counts):4.1f} | {np.mean(muon_counts):6.1f} +/- {np.std(muon_counts):4.1f}")

    print()

    # Print top-5 eigenvalue comparison (mean across seeds)
    print(f"  TOP-5 EIGENVALUE COMPARISON (mean across {NUM_SEEDS} seeds):")
    sgd_mean_eigs = np.mean(sgd_all_eigs, axis=0)
    muon_mean_eigs = np.mean(muon_all_eigs, axis=0)
    for k in range(min(5, len(sgd_mean_eigs))):
        print(f"    lambda_{k+1}: SGD = {sgd_mean_eigs[k]:.6e}, Muon = {muon_mean_eigs[k]:.6e}, ratio = {muon_mean_eigs[k]/sgd_mean_eigs[k]:.4f}" if abs(sgd_mean_eigs[k]) > 1e-15 else f"    lambda_{k+1}: SGD = {sgd_mean_eigs[k]:.6e}, Muon = {muon_mean_eigs[k]:.6e}")
    print()

    return all_results, ratios

In [ ]:
def main():
    print()
    print_separator('#')
    print("  EXPERIMENT 2.10: ANTI-SHARPNESS — Muon finds FLATTER minima")
    print("  Despite projecting onto sharp Hessian directions")
    print_separator('#')
    print()
    print(f"  Config: {NUM_SEEDS} seeds, max {NUM_STEPS} steps, convergence threshold = {CONVERGENCE_FACTOR}")
    print(f"  SGD lr = {LR_SGD}, Muon lr = {LR_MUON}, momentum = {MOMENTUM}")
    print(f"  Hessian computed via central finite differences (eps = {HESSIAN_EPS})")
    print()
    print(f"  THEORETICAL BACKGROUND:")
    print(f"    Deep linear net W_eff = W_L ... W_1 has gauge symmetry GL(d)^(L-1)")
    print(f"    For d={DIM}, each GL(d) contributes d^2={DIM**2} gauge dimensions")
    print(f"    2-layer: {1 * DIM**2} gauge dims / {2 * DIM**2} total = {1*DIM**2/(2*DIM**2)*100:.0f}% gauge-flat")
    print(f"    3-layer: {2 * DIM**2} gauge dims / {3 * DIM**2} total = {2*DIM**2/(3*DIM**2)*100:.0f}% gauge-flat")
    print()

    # ---- Part A: 2-layer net ----
    results_2layer, ratios_2layer = run_experiment_suite(
        num_layers=2, dim=DIM, label="PART A")

    # ---- Part B: 3-layer net ----
    results_3layer, ratios_3layer = run_experiment_suite(
        num_layers=3, dim=DIM, label="PART B")

    # ---- Final Summary ----
    print_separator('#')
    print("  FINAL COMPARATIVE SUMMARY")
    print_separator('#')
    print()
    print(f"  {'Metric':<28s} | {'2-layer Muon/SGD':<20s} | {'3-layer Muon/SGD':<20s}")
    print(f"  {'-'*28}-+-{'-'*20}-+-{'-'*20}")

    metric_labels = {
        'trace': 'tr(H) ratio',
        'lambda_max': 'lambda_max ratio',
        'eff_rank': 'Eff. rank ratio',
        'gauge_flat': 'Gauge-flat ratio',
        'kappa': 'Condition # ratio',
    }

    for m in ['trace', 'lambda_max', 'eff_rank', 'gauge_flat', 'kappa']:
        r2 = ratios_2layer[m]
        r3 = ratios_3layer[m]
        r2s = f"{r2:.4f}" if r2 != np.inf else "N/A"
        r3s = f"{r3:.4f}" if r3 != np.inf else "N/A"
        print(f"  {metric_labels[m]:<28s} | {r2s:<20s} | {r3s:<20s}")

    print()

    # Depth scaling analysis
    print(f"  DEPTH SCALING ANALYSIS:")
    t2 = ratios_2layer['trace']
    t3 = ratios_3layer['trace']
    print(f"    2-layer trace ratio: {t2:.4f}")
    print(f"    3-layer trace ratio: {t3:.4f}")
    if t3 < t2:
        print(f"    Change with depth: {((t2 - t3)/t2)*100:.1f}% reduction => anti-sharpness INCREASES with depth")
        print(f"    This is consistent with the gauge-fixing hypothesis: more gauge directions => stronger effect")
    elif t3 > t2:
        print(f"    Change with depth: {((t3 - t2)/t2)*100:.1f}% increase => anti-sharpness DECREASES with depth")
        print(f"    This is INCONSISTENT with the simple gauge-fixing prediction")
    else:
        print(f"    No change with depth")
    print()

    # Overall verdict
    print_separator('*')
    print("  OVERALL VERDICT:")
    print(f"    2-layer: tr(H_Muon)/tr(H_SGD) = {t2:.4f}  {'< 0.5 CONFIRMED' if t2 < 0.5 else '< 1.0 PARTIAL' if t2 < 1.0 else '>= 1.0 REJECTED'}")
    print(f"    3-layer: tr(H_Muon)/tr(H_SGD) = {t3:.4f}  {'< 0.5 CONFIRMED' if t3 < 0.5 else '< 1.0 PARTIAL' if t3 < 1.0 else '>= 1.0 REJECTED'}")
    print()
    if t3 < t2:
        print("    Deeper net shows STRONGER anti-sharpness effect (as predicted: more gauge directions)")
    else:
        print("    Deeper net does NOT show stronger anti-sharpness effect")
    print_separator('*')
    print()

In [ ]:
if __name__ == '__main__':
    main()

## Interpreting the Results

### What each outcome means for the theory

**If tr(H_Muon)/tr(H_SGD) < 0.5 (Hypothesis Confirmed):**
This would provide strong evidence that Muon's Newton-Schulz orthogonalization actively steers the optimizer away from sharp minima. The mechanism is: by equalizing step sizes across all singular directions, Muon moves aggressively through sharp valleys (where the gradient would normally be large and cause SGD to settle) and reaches the wider basins beyond. This is the "escape through sharpness" mechanism.

**If 0.5 < ratio < 1.0 (Partial Support):**
Muon finds flatter minima, but the effect is not as dramatic as the 2x threshold. This could mean: (a) the anti-sharpness effect exists but is moderate in these small networks, or (b) the learning rate difference between SGD and Muon partially confounds the comparison (higher LR generally correlates with flatter minima due to the "catapult" effect).

**If ratio >= 1.0 (Hypothesis Rejected):**
Muon does NOT find flatter minima despite projecting onto sharp directions. This would suggest that the Hessian-alignment result from Experiment 1.3b-ii-A does not translate into an anti-sharpness benefit, and the mechanism needs re-examination.

### Depth scaling prediction

The 3-layer net has **twice as many gauge directions** as the 2-layer net (32 vs 16 out of the total parameter count). If Muon's anti-sharpness effect operates specifically through gauge directions, the effect should be **stronger** in the 3-layer case. If it is not, the anti-sharpness mechanism may be independent of gauge structure and instead related to the general spectral equalization property of Newton-Schulz.

### Caveats

1. **Learning rate confound**: Muon uses 2x the learning rate of SGD. Higher LR alone can lead to flatter minima (the implicit regularization of large learning rates is well-documented). A definitive test would match effective step sizes.
2. **Small network size**: 32-48 parameters is far from realistic scale. The gauge structure of deep linear nets is well-understood theoretically, but may not generalize to nonlinear architectures.
3. **Convergence quality**: If one optimizer converges to a significantly lower loss than the other, the Hessian comparison is not truly apples-to-apples — the optimizers may be at different points on the loss landscape with inherently different curvatures.

## Conclusions

### Summary of Experiment 2.10

This experiment tested whether Muon's Newton-Schulz orthogonalization leads to convergence at **flatter minima** (lower Hessian trace) compared to SGD with momentum, despite prior evidence (Experiment 1.3b-ii-A) that Muon projects more aggressively onto sharp Hessian directions during training.

### Key Findings

1. **Primary hypothesis** ($\text{tr}(H_{\text{Muon}}) < 0.5 \cdot \text{tr}(H_{\text{SGD}})$): Check the printed verdict above. The trace ratio directly measures whether Muon's "escape through sharpness" mechanism produces measurably flatter converged solutions.

2. **Eigenvalue spectrum structure**: The binned eigenvalue analysis reveals whether the flatness difference (if any) comes from (a) uniformly smaller eigenvalues, (b) more near-zero eigenvalues (gauge-flat directions), or (c) reduction in the top eigenvalues only. Each pattern implies a different mechanism.

3. **Depth scaling**: Comparing 2-layer vs 3-layer results tests whether the anti-sharpness effect scales with the gauge dimension. A stronger effect at greater depth would support the RG gauge-fixing interpretation; no depth dependence would suggest a more generic spectral equalization mechanism.

### Implications for the Muon-as-Gauge-Fixing Theory

- If Muon consistently finds flatter minima AND the effect strengthens with depth, this supports the view that Newton-Schulz orthogonalization functions as a gauge-fixing operation that navigates the optimizer through the gauge manifold to wider basins.
- If the anti-sharpness effect exists but does NOT scale with depth, the mechanism is likely related to the general property of spectral democracy (equal step sizes in all singular directions) rather than gauge-specific dynamics.
- If no anti-sharpness effect is observed, the increased projection onto sharp Hessian directions may represent wasted computational effort rather than a beneficial exploration mechanism.

### Follow-up Experiments

- **LR-matched comparison**: Run with matched effective step sizes (e.g., both at LR=0.01) to disentangle the Newton-Schulz effect from the learning rate effect.
- **Nonlinear networks**: Test with ReLU activations to see if the anti-sharpness effect persists when gauge symmetries are broken by nonlinearity.
- **Trajectory analysis**: Track the Hessian trace *during training* (not just at convergence) to understand when Muon escapes sharp regions — early, mid, or late training.